# Quantum Reasoning Co-Processor — GPU Qubit Scaling (v2)
**Does quantum advantage grow with qubit count? Rigorous test.**

Author: Saikrishna Garikipati

> **Runtime → Run all** — GPU REQUIRED (T4/A100). ~3-4 hours total.

Fixes from v1:
- Signal density scales with dim (n_hidden = dim//8, not fixed at 4)
- Classical baseline is properly sized (fair comparison, not crippled)
- Per-seed error recovery (no lost results on crashes)
- Proven autograd pattern (no untested value_and_grad)
- Honest evaluation (skip experiment if both models are near random)

In [ ]:
##############################################################################
# STEP 1: SETUP & GPU VERIFICATION
##############################################################################
import os, sys, time
print('=' * 60)
print('STEP 1: SETUP & GPU VERIFICATION')
print('=' * 60)

!pip install -q pennylane pennylane-lightning 'pennylane-lightning[gpu]' autograd scipy 2>&1 | tail -3

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import autograd
import autograd.numpy as anp
import pennylane as qml
import pennylane.numpy as pnp

print(f'PennyLane {qml.__version__}')

# --- HARD GPU CHECK ---
try:
    test_dev = qml.device('lightning.gpu', wires=4)
    print(f'✓ lightning.gpu AVAILABLE')
    del test_dev
except Exception as e:
    print(f'✗ lightning.gpu NOT available: {e}')
    print()
    print('FATAL: This notebook REQUIRES GPU runtime.')
    print('Fix: Runtime → Change runtime type → GPU (T4 or A100)')
    print('Then: Restart runtime and run all again.')
    raise SystemExit('GPU required. Process killed.')

print('\nSetup complete. GPU ready.')

In [ ]:
##############################################################################
# STEP 2: DEFINE QUANTUM ATTENTION CIRCUIT
##############################################################################
print('=' * 60)
print('STEP 2: QUANTUM ATTENTION CIRCUIT')
print('=' * 60)

def build_attention_circuit(n_qubits, n_layers=2):
    """Build Q-K→V quantum attention circuit on GPU.
    Returns SINGLE SCALAR (mean PauliZ on V-register) for autograd compatibility.
    lightning.gpu + adjoint cannot return multiple expvals through autograd.grad().
    """
    assert n_qubits % 3 == 0
    qpr = n_qubits // 3
    q_wires = list(range(0, qpr))
    k_wires = list(range(qpr, 2 * qpr))
    v_wires = list(range(2 * qpr, 3 * qpr))

    dev = qml.device('lightning.gpu', wires=n_qubits)

    # Hamiltonian: average of PauliZ on V-register → single scalar output
    coeffs = [1.0 / qpr] * qpr
    obs = [qml.PauliZ(w) for w in v_wires]
    H = qml.Hamiltonian(coeffs, obs)

    @qml.qnode(dev, interface='autograd', diff_method='adjoint')
    def circuit(q_features, k_features, v_features, params):
        qml.AmplitudeEmbedding(q_features, wires=q_wires, normalize=True)
        qml.AmplitudeEmbedding(k_features, wires=k_wires, normalize=True)
        qml.AmplitudeEmbedding(v_features, wires=v_wires, normalize=True)
        idx = 0
        for layer in range(n_layers):
            for i in range(qpr):
                qml.CNOT(wires=[q_wires[i], k_wires[i]])
            for i in range(qpr):
                qml.CNOT(wires=[k_wires[i], v_wires[i]])
                qml.RY(params[idx], wires=v_wires[i])
                qml.CNOT(wires=[k_wires[i], v_wires[i]])
                idx += 1
            for i in range(qpr):
                qml.RY(params[idx], wires=v_wires[i])
                qml.RZ(params[idx + 1], wires=v_wires[i])
                idx += 2
            for i in range(qpr - 1):
                qml.CNOT(wires=[v_wires[i], v_wires[i + 1]])
            for i in range(qpr):
                qml.CNOT(wires=[q_wires[i], k_wires[i]])
        return qml.expval(H)

    n_params = n_layers * (qpr + qpr * 2)
    params = pnp.array(np.random.randn(n_params) * 0.5, requires_grad=True)
    print(f'  {n_qubits}q circuit: {n_params} params, register_dim={2**qpr}')
    print(f'  Returns: single scalar (mean PauliZ on V-register)')
    return circuit, params, qpr

# Quick verification
test_circuit, test_params, _ = build_attention_circuit(12)
q_test = np.random.randn(16)
out = test_circuit(q_test, q_test, q_test, test_params)
print(f'  12q forward pass OK: output={float(out):.4f} (scalar in [-1,1])')
print('Circuit definition ready.')

In [ ]:
##############################################################################
# STEP 3: DEFINE CLASSICAL BASELINE (FAIR — properly sized)
##############################################################################
print('=' * 60)
print('STEP 3: CLASSICAL BASELINE (fair capacity)')
print('=' * 60)

def build_classical_model(input_dim, n_hidden_task):
    """
    Classical MLP sized FAIRLY for the task — not crippled to match quantum param count.
    Given the task has n_hidden_task XOR dimensions to detect in input_dim features,
    the classical model gets enough hidden units to have a fair shot.
    Hidden size = 2 * n_hidden_task * 3 (enough to learn pairwise interactions).
    """
    hidden = max(16, n_hidden_task * 6)
    W1 = pnp.array(np.random.randn(input_dim * 3, hidden) * 0.1, requires_grad=True)
    b1 = pnp.array(np.zeros(hidden), requires_grad=True)
    W2 = pnp.array(np.random.randn(hidden, hidden // 2) * 0.1, requires_grad=True)
    b2 = pnp.array(np.zeros(hidden // 2), requires_grad=True)
    W3 = pnp.array(np.random.randn(hidden // 2, 1) * 0.1, requires_grad=True)
    b3 = pnp.array(np.zeros(1), requires_grad=True)
    total = W1.size + b1.size + W2.size + b2.size + W3.size + b3.size
    print(f'  Classical MLP: {total} params (hidden={hidden}, 3 layers)')
    return [W1, b1, W2, b2, W3, b3]

def classical_forward(q, k, v, params):
    W1, b1, W2, b2, W3, b3 = params
    x = anp.concatenate([q, k, v])
    h = anp.maximum(x @ W1 + b1, 0)
    h = anp.maximum(h @ W2 + b2, 0)
    out = anp.tanh(h @ W3 + b3)
    return out[0]  # scalar

print('Classical model defined (3-layer MLP, fairly sized).')

In [ ]:
##############################################################################
# STEP 4: DATA GENERATION (signal scales with dimension)
##############################################################################
print('=' * 60)
print('STEP 4: XOR-SIGN CLASSIFICATION TASK (scaled signal)')
print('=' * 60)

def generate_xor_sign_data(dim, n_samples, seed=42):
    """
    Task: Predict XOR of sign-agreements between Q and K on hidden dims.
    FIXED: n_hidden scales with dim to maintain constant signal density (~12.5%).
    This ensures the task stays learnable at all qubit scales.
    """
    rng = np.random.RandomState(seed)
    n_hidden = max(2, dim // 8)  # ~12.5% of dims carry signal
    hidden_dims = sorted(rng.choice(dim, n_hidden, replace=False).tolist())
    Q = rng.randn(n_samples, dim).astype(np.float64)
    K = rng.randn(n_samples, dim).astype(np.float64)
    V = rng.randn(n_samples, dim).astype(np.float64)
    agreements = np.array([(np.sign(Q[:, i]) == np.sign(K[:, i])).astype(int)
                           for i in hidden_dims])
    labels = np.bitwise_xor.reduce(agreements, axis=0)
    return Q, K, V, labels, hidden_dims

# Verify scaling
for nq in [18, 21, 24]:
    dim = 2 ** (nq // 3)
    _, _, _, y, hd = generate_xor_sign_data(dim, 200, seed=42)
    print(f'  {nq}q (dim={dim}): n_hidden={len(hd)}, balance={y.mean():.2f}')
print('Data generation ready (signal density constant across scales).')

In [ ]:
##############################################################################
# STEP 5: TRAINING FUNCTIONS (proven pattern + error recovery)
##############################################################################
print('=' * 60)
print('STEP 5: TRAINING INFRASTRUCTURE')
print('=' * 60)

def train_quantum_circuit(circuit, params, Q, K, V, y, epochs=30, lr=0.01, batch_size=32):
    """Train quantum circuit. Uses proven autograd.grad pattern from Run 10."""
    n = len(Q)
    for epoch in range(epochs):
        idx = np.random.permutation(n)
        epoch_loss = 0.0
        nb = 0
        for start in range(0, min(n, 200) - batch_size + 1, batch_size):
            batch = idx[start:start + batch_size]
            def loss_fn(p):
                total = 0.0
                for j in batch:
                    score = circuit(Q[j], K[j], V[j], p)
                    pred = (score + 1) / 2
                    t = float(y[j])
                    total = total - (t * anp.log(pred + 1e-10) + (1-t) * anp.log(1-pred + 1e-10))
                return total / len(batch)
            grads = autograd.grad(loss_fn)(params)
            epoch_loss += float(loss_fn(params))
            params = pnp.array(params - lr * np.array(grads), requires_grad=True)
            nb += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            acc = eval_quantum(circuit, params, Q[:min(100,n)], K[:min(100,n)], V[:min(100,n)], y[:min(100,n)])
            print(f'    Epoch {epoch+1:2d}: loss={epoch_loss/max(nb,1):.4f}, acc={acc:.1%}')
    return params

def train_classical_model(params, Q, K, V, y, epochs=30, lr=0.01, batch_size=32):
    """Train classical MLP. Uses proven autograd.grad pattern."""
    n = len(Q)
    for epoch in range(epochs):
        idx = np.random.permutation(n)
        epoch_loss = 0.0
        nb = 0
        for start in range(0, min(n, 200) - batch_size + 1, batch_size):
            batch = idx[start:start + batch_size]
            def loss_fn(flat):
                sizes = [p.size for p in params]
                shapes = [p.shape for p in params]
                ps = []
                offset = 0
                for s, sh in zip(sizes, shapes):
                    ps.append(flat[offset:offset+s].reshape(sh))
                    offset += s
                total = 0.0
                for j in batch:
                    score = classical_forward(Q[j], K[j], V[j], ps)
                    pred = (score + 1) / 2
                    t = float(y[j])
                    total = total - (t * anp.log(pred + 1e-10) + (1-t) * anp.log(1-pred + 1e-10))
                return total / len(batch)
            flat = np.concatenate([np.array(p).flatten() for p in params])
            grads = autograd.grad(loss_fn)(flat)
            epoch_loss += float(loss_fn(flat))
            flat = flat - lr * np.array(grads)
            offset = 0
            for k_idx, p in enumerate(params):
                s = p.size
                params[k_idx] = pnp.array(flat[offset:offset+s].reshape(p.shape), requires_grad=True)
                offset += s
            nb += 1
        if (epoch + 1) % 10 == 0 or epoch == 0:
            acc = eval_classical(params, Q[:min(100,n)], K[:min(100,n)], V[:min(100,n)], y[:min(100,n)])
            print(f'    Epoch {epoch+1:2d}: loss={epoch_loss/max(nb,1):.4f}, acc={acc:.1%}')
    return params

def eval_quantum(circuit, params, Q, K, V, y):
    correct = sum(1 for j in range(len(Q))
                  if (1 if float(circuit(Q[j], K[j], V[j], params)) > 0 else 0) == int(y[j]))
    return correct / len(Q)

def eval_classical(params, Q, K, V, y):
    correct = sum(1 for j in range(len(Q))
                  if (1 if float(classical_forward(Q[j], K[j], V[j], params)) > 0 else 0) == int(y[j]))
    return correct / len(Q)

def run_single_experiment(n_qubits, seed, epochs=30, n_train=400, n_test=100):
    """Run one (quantum vs classical) comparison. Returns result dict or None on failure."""
    dim = 2 ** (n_qubits // 3)
    n_hidden_task = max(2, dim // 8)

    try:
        np.random.seed(seed)
        Q, K, V, y, hd = generate_xor_sign_data(dim, n_train + n_test, seed=seed)

        # Quantum
        t0 = time.perf_counter()
        circ, params, _ = build_attention_circuit(n_qubits, n_layers=2)
        params = train_quantum_circuit(circ, params, Q[:n_train], K[:n_train], V[:n_train], y[:n_train], epochs=epochs)
        q_acc = eval_quantum(circ, params, Q[n_train:], K[n_train:], V[n_train:], y[n_train:])
        q_time = time.perf_counter() - t0

        # Classical (fair baseline)
        t0 = time.perf_counter()
        cp = build_classical_model(dim, n_hidden_task)
        cp = train_classical_model(cp, Q[:n_train], K[:n_train], V[:n_train], y[:n_train], epochs=epochs)
        c_acc = eval_classical(cp, Q[n_train:], K[n_train:], V[n_train:], y[n_train:])
        c_time = time.perf_counter() - t0

        result = {
            'n_qubits': n_qubits, 'seed': seed, 'dim': dim,
            'n_hidden': n_hidden_task,
            'q_acc': q_acc, 'c_acc': c_acc,
            'advantage': q_acc - c_acc,
            'q_time': q_time, 'c_time': c_time,
        }
        print(f'    ✓ seed={seed}: Q={q_acc:.1%} C={c_acc:.1%} adv={result["advantage"]*100:+.1f}% ({q_time:.0f}s)')
        return result

    except Exception as e:
        print(f'    ✗ seed={seed}: FAILED — {e}')
        return None

print('Training functions ready (with per-seed error recovery).')

In [ ]:
##############################################################################
# STEP 6: RUN ALL QUBIT SCALES (18q × 5 seeds, 21q × 3 seeds, 24q × 2 seeds)
##############################################################################
print('=' * 60)
print('STEP 6: QUBIT SCALING EXPERIMENTS')
print('=' * 60)

RESULTS = {}
SEEDS = [42, 99, 137, 256, 314]

EXPERIMENTS = [
    {'n_qubits': 18, 'seeds': SEEDS[:5], 'epochs': 30, 'label': '18q (5 seeds, dim=64, 8 hidden)'},
    {'n_qubits': 21, 'seeds': SEEDS[:3], 'epochs': 30, 'label': '21q (3 seeds, dim=128, 16 hidden)'},
    {'n_qubits': 24, 'seeds': SEEDS[:2], 'epochs': 20, 'label': '24q (2 seeds, dim=256, 32 hidden)'},
]

for exp in EXPERIMENTS:
    nq = exp['n_qubits']
    dim = 2 ** (nq // 3)
    n_hidden = max(2, dim // 8)
    print(f'\n{"="*50}')
    print(f'  {exp["label"]}')
    print(f'  register_dim={dim}, n_hidden_task={n_hidden}, quantum_params={2*(nq//3 + (nq//3)*2)}')
    print(f'{"="*50}')

    seed_results = []
    for seed in exp['seeds']:
        result = run_single_experiment(nq, seed, epochs=exp['epochs'])
        if result is not None:
            seed_results.append(result)

    if seed_results:
        q_accs = [r['q_acc'] for r in seed_results]
        c_accs = [r['c_acc'] for r in seed_results]
        mean_q = np.mean(q_accs)
        mean_c = np.mean(c_accs)
        adv = mean_q - mean_c
        RESULTS[nq] = {
            'results': seed_results,
            'q_accs': q_accs, 'c_accs': c_accs,
            'mean_q': mean_q, 'mean_c': mean_c,
            'advantage': adv, 'n_seeds': len(seed_results),
        }
        # Honesty check: if both near random, flag it
        both_random = (abs(mean_q - 0.5) < 0.05 and abs(mean_c - 0.5) < 0.05)
        flag = ' ⚠️ BOTH NEAR RANDOM' if both_random else ''
        print(f'\n  {nq}q SUMMARY: Q={mean_q:.1%}±{np.std(q_accs):.1%} vs C={mean_c:.1%}±{np.std(c_accs):.1%} → adv={adv*100:+.1f}%{flag}')
    else:
        print(f'\n  {nq}q: ALL SEEDS FAILED')
        RESULTS[nq] = {'failed': True}

print('\n\nAll experiments complete.')

In [ ]:
##############################################################################
# STEP 7: DEEPER CIRCUIT VARIANT (18q, n_layers=4 instead of 2)
##############################################################################
print('=' * 60)
print('STEP 7: DEEPER CIRCUIT (18q, 4 layers vs 2 layers)')
print('=' * 60)
print('Testing if more circuit depth (more params) helps at 18q.')
print('This addresses the barren plateau concern — maybe 2 layers is too shallow.')

def build_deep_circuit(n_qubits, n_layers=4):
    """Same architecture but deeper — more trainable parameters."""
    assert n_qubits % 3 == 0
    qpr = n_qubits // 3
    q_wires = list(range(0, qpr))
    k_wires = list(range(qpr, 2 * qpr))
    v_wires = list(range(2 * qpr, 3 * qpr))

    dev = qml.device('lightning.gpu', wires=n_qubits)
    coeffs = [1.0 / qpr] * qpr
    obs = [qml.PauliZ(w) for w in v_wires]
    H = qml.Hamiltonian(coeffs, obs)

    @qml.qnode(dev, interface='autograd', diff_method='adjoint')
    def circuit(q_features, k_features, v_features, params):
        qml.AmplitudeEmbedding(q_features, wires=q_wires, normalize=True)
        qml.AmplitudeEmbedding(k_features, wires=k_wires, normalize=True)
        qml.AmplitudeEmbedding(v_features, wires=v_wires, normalize=True)
        idx = 0
        for layer in range(n_layers):
            for i in range(qpr):
                qml.CNOT(wires=[q_wires[i], k_wires[i]])
            for i in range(qpr):
                qml.CNOT(wires=[k_wires[i], v_wires[i]])
                qml.RY(params[idx], wires=v_wires[i])
                qml.CNOT(wires=[k_wires[i], v_wires[i]])
                idx += 1
            for i in range(qpr):
                qml.RY(params[idx], wires=v_wires[i])
                qml.RZ(params[idx + 1], wires=v_wires[i])
                idx += 2
            for i in range(qpr - 1):
                qml.CNOT(wires=[v_wires[i], v_wires[i + 1]])
            for i in range(qpr):
                qml.CNOT(wires=[q_wires[i], k_wires[i]])
        return qml.expval(H)

    n_params = n_layers * (qpr + qpr * 2)
    params = pnp.array(np.random.randn(n_params) * 0.3, requires_grad=True)
    print(f'  Deep {n_qubits}q circuit: {n_params} params ({n_layers} layers)')
    return circuit, params, qpr

# Run deep circuit experiment (18q, 3 seeds)
deep_results = []
for seed in SEEDS[:3]:
    try:
        np.random.seed(seed)
        dim = 64
        Q, K, V, y, hd = generate_xor_sign_data(dim, 500, seed=seed)
        print(f'\n  Seed {seed}:')

        t0 = time.perf_counter()
        circ, params, _ = build_deep_circuit(18, n_layers=4)
        params = train_quantum_circuit(circ, params, Q[:400], K[:400], V[:400], y[:400], epochs=30, lr=0.005)
        q_acc = eval_quantum(circ, params, Q[400:], K[400:], V[400:], y[400:])
        q_time = time.perf_counter() - t0
        deep_results.append({'q_acc': q_acc, 'time': q_time})
        print(f'    Deep 18q: {q_acc:.1%} ({q_time:.0f}s)')
    except Exception as e:
        print(f'    FAILED: {e}')

if deep_results:
    deep_mean = np.mean([r['q_acc'] for r in deep_results])
    shallow_mean = RESULTS.get(18, {}).get('mean_q', 0)
    print(f'\n  Deep (4 layers): {deep_mean:.1%}')
    print(f'  Shallow (2 layers): {shallow_mean:.1%}')
    print(f'  Depth benefit: {(deep_mean - shallow_mean)*100:+.1f}%')
    RESULTS['18_deep'] = {'mean_q': deep_mean, 'results': deep_results}

In [ ]:
##############################################################################
# STEP 8: FINAL REPORT
##############################################################################
print('=' * 60)
print('FINAL REPORT: QUBIT SCALING RESULTS')
print('=' * 60)
print()

# Summary table
print(f'{"Config":<14} {"Qubits":<8} {"Dim":<6} {"Hidden":<8} {"Quantum":<16} {"Classical":<16} {"Advantage":<12} {"Seeds":<6}')
print('-' * 86)
for key in sorted(k for k in RESULTS.keys() if isinstance(k, int)):
    r = RESULTS[key]
    if r.get('failed'):
        print(f'{"standard":<14} {key:<8} {2**(key//3):<6} {"-":<8} {"FAILED":<16} {"-":<16} {"-":<12} 0')
        continue
    dim = 2 ** (key // 3)
    n_hidden = max(2, dim // 8)
    ns = r['n_seeds']
    q_str = f"{r['mean_q']*100:.1f}%±{np.std(r['q_accs'])*100:.1f}%" if ns > 1 else f"{r['mean_q']*100:.1f}%"
    c_str = f"{r['mean_c']*100:.1f}%±{np.std(r['c_accs'])*100:.1f}%" if ns > 1 else f"{r['mean_c']*100:.1f}%"
    print(f'{"standard":<14} {key:<8} {dim:<6} {n_hidden:<8} {q_str:<16} {c_str:<16} {r["advantage"]*100:+.1f}%       {ns}')

if '18_deep' in RESULTS:
    r = RESULTS['18_deep']
    print(f'{"deep(4layer)":<14} {18:<8} {64:<6} {8:<8} {r["mean_q"]*100:.1f}%{"":11} {"-":<16} {"-":<12} {len(r["results"])}')

# Statistical test for 18q
print()
print('=' * 60)
print('STATISTICAL SIGNIFICANCE')
print('=' * 60)
for nq in sorted(k for k in RESULTS.keys() if isinstance(k, int)):
    r = RESULTS[nq]
    if r.get('failed') or r['n_seeds'] < 3:
        continue
    from scipy import stats
    t_stat, p_value = stats.ttest_rel(r['q_accs'], r['c_accs'])
    diff = np.array(r['q_accs']) - np.array(r['c_accs'])
    cohens_d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else 0
    sig = '★ YES' if p_value < 0.05 else 'no'
    print(f'  {nq}q ({r["n_seeds"]} seeds): mean_adv={np.mean(diff)*100:+.1f}%, p={p_value:.4f}, d={cohens_d:.2f} → {sig}')

# Scaling verdict
print()
print('=' * 60)
print('VERDICT')
print('=' * 60)
valid = {k: v for k, v in RESULTS.items() if isinstance(k, int) and not v.get('failed')}

if len(valid) >= 2:
    sorted_nq = sorted(valid.keys())
    advantages = [valid[nq]['advantage'] for nq in sorted_nq]
    both_random = [abs(valid[nq]['mean_q'] - 0.5) < 0.06 and abs(valid[nq]['mean_c'] - 0.5) < 0.06
                   for nq in sorted_nq]

    print(f'  Scaling: {", ".join(f"{nq}q={a*100:+.1f}%" for nq, a in zip(sorted_nq, advantages))}')
    print(f'  Near-random flags: {", ".join(f"{nq}q={'⚠️' if br else '✓'}" for nq, br in zip(sorted_nq, both_random))}')
    print()

    if all(both_random):
        print('  ✗ ALL EXPERIMENTS NEAR RANDOM — neither model learns the task at these scales')
        print('  → Task may be too hard, or both models need more epochs/capacity')
    elif any(a > 0.05 for a in advantages) and not all(both_random):
        any_sig = any(valid[nq]['n_seeds'] >= 3 and
                      stats.ttest_rel(valid[nq]['q_accs'], valid[nq]['c_accs'])[1] < 0.05
                      for nq in sorted_nq if valid[nq]['n_seeds'] >= 3)
        if any_sig:
            print('  ✓ STATISTICALLY SIGNIFICANT quantum advantage detected')
            print('  → Proceed to Month 2 (real quantum hardware)')
        else:
            print('  ~ Advantage observed but not statistically significant')
            print('  → Need more seeds or epochs to confirm')
    else:
        print('  ✗ No meaningful quantum advantage at any tested scale')
        print('  → Circuit architecture may need fundamental rethinking')
else:
    print('  Not enough completed experiments for verdict')

print()
print('Copyright (c) 2026 Saikrishna Garikipati. All Rights Reserved.')